# DAT09 · 미션1 보고 형식과 업무시간 확인하기 (학습자용)

- **예상 실습 시간**: 1.5시간 (전체 10개 미션 총 27시간 과정)
- **작업 폴더**: `DAT09_practice`
- **연결 파일**: 첫 셀에서 `DAT09_3.md`(또는 `DAT09.md`)를 연결하세요.

- 각 단계 함수(`m01_s1`~`m01_s5`)의 `TODO`를 직접 작성한 뒤 셀을 실행하세요.
- 실행 결과는 `outputs/M01/DAT09_M01_result.html`과 `step_01.csv`~`step_05.csv`에 저장됩니다.


In [ ]:
from pathlib import Path, PurePosixPath
import sys, io, json, re, zipfile, base64, hashlib, importlib
from IPython.display import display, HTML
import html
import pandas as pd
ROLE = 'student'
EXPECTED_SHA256 = 'a0adfd7b9a2416cd74af2fc99bb7978ad662eec8d9e40ce5b8a539977c2cb17e'
c = None

def connect_md(path):
    global c, ROOT, HERE
    path = Path(path)
    if path.name not in ('DAT09.md', 'DAT09_3.md'):
        raise ValueError('원자료 DAT09.md 또는 DAT09_3.md를 선택하세요. 정답코드 MD는 연결 파일이 아닙니다.')
    text = path.read_text(encoding='utf-8-sig')
    found = re.search(r'<!-- DAT09_RUNTIME_BUNDLE_B64:([A-Za-z0-9+/=]+) -->', text)
    if not found:
        raise ValueError('이 노트북용 원자료 블록이 없습니다. 함께 받은 DAT09.md를 사용하세요.')
    raw = base64.b64decode(found.group(1), validate=True)
    if hashlib.sha256(raw).hexdigest() != EXPECTED_SHA256:
        raise ValueError('자료와 노트북 버전이 다릅니다. 같은 묶음의 파일을 사용하세요.')
    ROOT = Path.cwd() / ('DAT09_reference' if ROLE == 'solution' else 'DAT09_practice')
    ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw)) as archive:
        for info in archive.infolist():
            name = PurePosixPath(info.filename)
            if name.is_absolute() or '..' in name.parts:
                raise ValueError('허용되지 않은 경로입니다.')
            target = ROOT.joinpath(*name.parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(info))
    sys.path.insert(0, str(ROOT))
    for name in list(sys.modules):
        if name in ['dat09_common', 'dat09_ui'] or name.startswith('src.m'):
            del sys.modules[name]
    import dat09_common
    globals().update({name: getattr(dat09_common, name) for name in dir(dat09_common) if not name.startswith('_')})
    import collections, itertools, queue, threading, tempfile, importlib.util, shutil
    globals().update({k:v for k,v in locals().items() if k in ['collections','itertools','re','queue','threading','tempfile','importlib','shutil']})
    from dat09_ui import bars, heatmap, line_chart, dashboard_fragment
    globals().update(bars=bars,heatmap=heatmap,line_chart=line_chart,dashboard_fragment=dashboard_fragment)
    c = dat09_common.context(ROOT)
    HERE = ROOT.parent / 'instructor'
    for candidate in [Path.cwd()/'instructor', Path.cwd(), ROOT.parent/'instructor']:
        if (candidate/'aar_key.json').is_file():
            HERE = candidate
            c['instructor_root'] = candidate
            break
    print('자료 연결 완료:', ROOT)
    print('10개 미션·50개 함수·27시간 과정입니다. 미션1부터 실행하세요.')
    print('TODO가 남아 있으면 해당 단계에서 멈춥니다. src에는 작성한 함수를 저장합니다.')
    return c

DATA_MD = next((Path(p) for p in ('DAT09.md', 'DAT09_3.md') if Path(p).is_file()), None)
if DATA_MD is not None:
    connect_md(DATA_MD)
else:
    print('DAT09_3.md(또는 DAT09.md)를 이 노트북과 같은 폴더에 두고 첫 셀을 다시 실행하세요.')
    print("다른 경로에 있다면 connect_md('/자료가_있는_경로/DAT09_3.md')를 실행하세요.")


## 미션1 보고 형식과 업무시간 확인하기

예상 1.5시간입니다.

#### 이번 미션의 목표

세 파일이 어떤 재고를 보고하는지 확인하고, 뒤의 코드 작성에서 지킬 변환 규칙과 개선 순서를 정합니다.

### 1단계 세 재고 보고의 항목과 뜻 비교하기

#### 이 단계에서 하는 일

보급·정비·탄약이라는 파일 이름만 보고 수량을 합치면 같은 재고를 세 번 더할 수 있습니다. 이번 자료에서는 세 출처가 같은 거점과 품목의 보유 수량을 각각 보고합니다. 먼저 같은 대상을 비교할 수 있는지 확인해야 합니다.

#### 시작 전 확인

노트북 첫 셀로 자료를 연결한 상태에서 시작합니다. 노트북의 m01_s1(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

data/supply.csv, data/maintenance.json, data/ammunition.tsv와 config/source_contracts.json을 사용합니다. source_contracts의 field_mapping은 출처별 항목 이름을 공통 이름으로 바꾸는 표이며, refresh_minutes는 정해 놓은 갱신 간격입니다.

#### 작업 순서

- CSV는 쉼표, TSV는 탭으로 항목을 나누어 읽고, JSON은 records 안의 보고 목록을 읽으세요. 파일 이름만 목록에 올리지 말고 실제 보고 행을 읽습니다.

- 파일별로 출처, 형식, 행 수, 항목 이름, 첫 보고의 코드·수량·단위·날짜 표기, 갱신 간격을 한 표에 모으세요. SHA256 값도 남깁니다. SHA256은 나중에 같은 파일을 사용했는지 비교하는 식별값입니다.

- 동일한 거점·품목·기준시각의 보고를 세 출처에서 하나씩 찾아 비교하세요. 보고된 수량의 뜻은 on_hand, 즉 그 시점의 보유 수량입니다. 같은 재고의 재보고를 합산하지 않는 이유를 기록하세요.


#### 저장할 결과

source_spec.csv와 세 파일의 표본을 남깁니다. 표본은 다음 단계에서 다시 읽거나 c['sample_sources']에 보관해 사용할 수 있습니다.

#### 직접 확인하는 방법

파일마다 720행인지, 전체가 2,160행인지 직접 세세요. 예를 들어 동일 재고가 세 파일에서 각각 80EA로 보고돼도 재고는 240EA가 되지 않습니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M01/step_01.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M01_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 2단계 출처별 코드를 같은 품목과 거점으로 연결하기

#### 이 단계에서 하는 일

같은 품목도 출처에 따라 코드가 다릅니다. 공통 코드를 붙이되 원래 코드를 잃지 않아야 잘못 연결한 행을 되짚을 수 있습니다. 또한 무엇을 하나의 보고로 볼지 정해야 중복을 판별할 수 있습니다.

#### 시작 전 확인

미션1의 1단계까지 작성·확인한 결과를 사용합니다. 노트북의 m01_s2(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

data/catalog.json의 mapping, items, sites와 1단계의 비교표를 사용합니다. mapping은 각 출처의 원래 품목·거점 코드를 공통 코드로 연결합니다.

#### 작업 순서

- 출처별 원래 품목 코드와 거점 코드를 공통 item_id, site_id로 바꾼 표를 만드세요. 출처마다 품목 20개와 거점 6개의 조합 120개, 전체 360개 조합을 검사합니다.

- 공통 코드에서 출처별 원래 코드로 다시 돌아오는 표도 만드세요. 서로 다른 원래 품목이 잘못 하나로 합쳐지거나, 돌아온 코드가 달라진 조합을 따로 셉니다.

- 재고 이력을 구별하는 조합은 site_id·item_id·metric·as_of로 정하세요. 출처의 관측 보고를 구별할 때는 source를 더합니다. 실시간 현재값은 source·site_id·item_id·metric별로 유지하고 버전을 비교합니다. 이 세 구분을 key_contract.json에 적으세요.


#### 저장할 결과

360개 변환·복원 결과를 단계 반환 표로 남기고, 식별 기준과 충돌 건수를 key_contract.json에 저장합니다.

#### 직접 확인하는 방법

360개 조합에서 변환한 뒤 복원한 코드가 모두 같아야 합니다. 서로 다른 기준시각의 보고가 하나의 이력으로 덮어써지지 않는지도 확인하세요.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M01/step_02.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M01_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 3단계 업무시간을 계산하고 지연이 큰 단계 찾기

#### 이 단계에서 하는 일

자동화 효과를 말하려면 현재 어디에서 시간이 드는지부터 알아야 합니다. 사람이 작업하는 시간과 다음 처리를 기다리는 시간은 개선 방법이 다릅니다. 이번 파일의 시간은 실제 부대 실측값이 아닌 실습용 가정입니다.

#### 시작 전 확인

미션1의 2단계까지 작성·확인한 결과를 사용합니다. 노트북의 m01_s3(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

data/workflow.csv의 12개 업무 단계와 processing_min, waiting_min을 사용합니다. 두 열의 단위는 분입니다.

#### 작업 순서

- 각 단계의 처리시간과 대기시간을 숫자로 읽고, 단계별 합계와 전체 합계를 계산하세요. 전체 처리시간과 전체 대기시간도 구분해 적습니다.

- 처리시간이 30분 이상이거나 대기시간이 30분 이상이면 해당 단계를 지연이 큰 단계로 표시하세요. 두 조건에 모두 해당해도 단계 수는 한 번만 셉니다.

- 단계별 합계 시간을 막대그래프로 그리세요. 개선 후 전체 시간을 50분으로 가정하고 (현재시간-개선후시간)/현재시간으로 절감률을 구합니다. 표와 그래프에 실습용 가정임을 표시하세요.


#### 저장할 결과

workflow_comparison.csv에 12단계의 두 시간과 합계·판정값을 저장하고, workflow_assumptions.json에 전체 합계·개선 가정·절감률을 저장합니다.

#### 직접 확인하는 방법

처리 210분, 대기 340분, 합계 550분과 지연이 큰 단계 8개를 확인하세요. 550분에서 50분으로 줄인다는 가정의 절감률은 약 90.91%입니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M01/step_03.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M01_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 4단계 현재 환경을 평가하고 먼저 고칠 일 정하기

#### 이 단계에서 하는 일

뒤의 미션을 모두 끝낸 것처럼 점수를 주면 무엇이 개선됐는지 알 수 없습니다. 지금까지 확보한 근거만으로 현재 상태를 평가하고, 다음 작업의 순서를 정합니다.

#### 시작 전 확인

미션1의 3단계까지 작성·확인한 결과를 사용합니다. 노트북의 m01_s4(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

1~3단계의 비교표·코드 검사·업무시간 결과와 config/actor_registry.json을 사용합니다. 평가 영역은 파일 연결, 코드 표준, 자료 품질, 갱신 적시성, 권한과 책임, 다시 실행하기의 여섯 가지입니다.

#### 작업 순서

- 각 영역에 현재 확인한 사실, 부족한 점, 담당자, 다음 작업을 적으세요. 근거가 없으면 0점, 문서나 수동 확인만 있으면 1점, 별도 시험으로 정상·오류 상황까지 확인했으면 2점을 줍니다.

- 점수 옆에 확인한 파일·행 또는 시험 이름을 적으세요. 앞으로 만들 계획만으로 점수를 올리지 않습니다. 코드를 한 번 실행했다는 사실만으로 오류 대응까지 검증한 것으로 보지 않습니다.

- 가장 먼저 고칠 작업 세 개를 고르고 순서, 담당자, 선행 작업, 완료를 확인할 조건을 적으세요. 예를 들어 코드 변환을 먼저 끝내야 품질 집계를 할 수 있다는 관계를 드러냅니다.


#### 저장할 결과

environment_baseline.csv에 여섯 영역의 평가를, improvement_backlog.json에 우선 작업 세 개를 저장합니다.

#### 직접 확인하는 방법

여섯 영역이 빠짐없이 있고 점수는 0~2 범위인지 확인하세요. 세 작업의 완료 조건은 “개선한다”가 아니라 어떤 파일이나 시험이 나오면 끝나는지로 적습니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M01/step_04.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M01_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 5단계 다음 담당자가 사용할 변환 규칙 작성하기

#### 이 단계에서 하는 일

이 문서는 다음 미션에서 파일을 읽고 값을 바꾸는 기준입니다. “표준화한다”는 말만으로는 어떤 값을 어디까지 바꿔도 되는지 알 수 없습니다. 다른 사람이 이 문서만 보고 같은 처리를 할 수 있게 씁니다.

#### 시작 전 확인

미션1의 4단계까지 작성·확인한 결과를 사용합니다. 노트북의 m01_s5(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

세 파일의 비교표, catalog.json, source_contracts.json과 앞 단계의 식별 기준을 사용합니다. 새 규칙을 임의로 추가하지 말고 제공 자료에서 확인한 기준을 옮깁니다.

#### 작업 순서

- spec.md에 입력 파일과 공통 항목의 대응, 품목·거점 코드표, 재고 이력과 출처별 관측의 식별 기준을 적으세요.

- 기본 단위 EA, 등록된 BOX의 10EA 환산, 시간대가 있는 시각의 UTC 변환, 60분 초과 지연 표시 방법을 예시와 함께 쓰세요. 원래 코드·값·단위·시각은 보존합니다.

- 결측 수량은 0으로 만들지 않고, 미등록 코드·단위·잘못된 날짜는 확인할 때까지 보류 또는 격리한다고 적으세요. 다른 두 출처가 같아도 보완 후보일 뿐 승인값이 아님을 명시합니다.

- 동료에게 stock_amount라는 새 항목과 출고수량이라는 새 항목을 각각 제시하세요. 전자는 재고와 같은 뜻인지 확인한 뒤 등록하고, 후자는 재고량으로 바로 연결하지 않는 이유를 검토 기록에 남깁니다.


#### 저장할 결과

spec.md와 검토자·검토일·질문·답변·수정 사항을 적은 검토 기록을 저장합니다. 혼자 진행하면 교관이 동료 검토를 대신할 수 있습니다.

#### 직접 확인하는 방법

파일·항목·코드·단위·시각·결측·지연·후보·승인의 기준이 모두 있는지 체크하세요. “적절히 처리한다”처럼 선택을 남기는 문장은 구체적인 조건으로 바꿉니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M01/step_05.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M01_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.



In [ ]:
if c is None:
    raise RuntimeError("첫 셀에서 DAT09_3.md(또는 DAT09.md)를 먼저 연결하세요.")

# STEP 01 세 재고 보고의 항목과 뜻 비교하기
# 할 일: CSV는 쉼표, TSV는 탭으로 항목을 나누어 읽고, JSON은 records 안의 보고 목록을 읽으세요. 파일 이름만 목록에 올리지 말고 실제 보고 행을 읽습니다. 파일별로 출처, 형식, 행 수, 항목 이름, 첫 보고의 코드·수량·단위·날짜 표기, 갱신 간격을 한 표에 모으세요. SHA256 값도 남깁니다. SHA256은 나중에 같은 파일을 사용했는지 비교하는 식별값입니다. 동일한 거점·품목·기준시각의 보고를 세 출처에서 하나씩 찾아 비교하세요. 보고된 수량의 뜻은 on_hand, 즉 그 시점의 보유 수량입니다. 같은 재고의 재보고를 합산하지 않는 이유를 기록하세요.
# 확인: 파일마다 720행인지, 전체가 2,160행인지 직접 세세요. 예를 들어 동일 재고가 세 파일에서 각각 80EA로 보고돼도 재고는 240EA가 되지 않습니다.
def m01_s1(c):
    """세 재고 보고의 항목과 뜻 비교하기.

    읽을 작업값: checks, contract, root
    저장할 작업값: 근거 파일과 반환 표
    결과 폴더: folder(c, 1)
    """
    return TODO(1, 1)

# STEP 02 출처별 코드를 같은 품목과 거점으로 연결하기
# 할 일: 출처별 원래 품목 코드와 거점 코드를 공통 item_id, site_id로 바꾼 표를 만드세요. 출처마다 품목 20개와 거점 6개의 조합 120개, 전체 360개 조합을 검사합니다. 공통 코드에서 출처별 원래 코드로 다시 돌아오는 표도 만드세요. 서로 다른 원래 품목이 잘못 하나로 합쳐지거나, 돌아온 코드가 달라진 조합을 따로 셉니다. 재고 이력을 구별하는 조합은 site_id·item_id·metric·as_of로 정하세요. 출처의 관측 보고를 구별할 때는 source를 더합니다. 실시간 현재값은 source·site_id·item_id·metric별로 유지하고 버전을 비교합니다. 이 세 구분을 key_contract.json에 적으세요.
# 확인: 360개 조합에서 변환한 뒤 복원한 코드가 모두 같아야 합니다. 서로 다른 기준시각의 보고가 하나의 이력으로 덮어써지지 않는지도 확인하세요.
def m01_s2(c):
    """출처별 코드를 같은 품목과 거점으로 연결하기.

    읽을 작업값: catalog, checks
    저장할 작업값: 근거 파일과 반환 표
    결과 폴더: folder(c, 1)
    """
    return TODO(1, 2)

# STEP 03 업무시간을 계산하고 지연이 큰 단계 찾기
# 할 일: 각 단계의 처리시간과 대기시간을 숫자로 읽고, 단계별 합계와 전체 합계를 계산하세요. 전체 처리시간과 전체 대기시간도 구분해 적습니다. 처리시간이 30분 이상이거나 대기시간이 30분 이상이면 해당 단계를 지연이 큰 단계로 표시하세요. 두 조건에 모두 해당해도 단계 수는 한 번만 셉니다. 단계별 합계 시간을 막대그래프로 그리세요. 개선 후 전체 시간을 50분으로 가정하고 (현재시간-개선후시간)/현재시간으로 절감률을 구합니다. 표와 그래프에 실습용 가정임을 표시하세요.
# 확인: 처리 210분, 대기 340분, 합계 550분과 지연이 큰 단계 8개를 확인하세요. 550분에서 50분으로 줄인다는 가정의 절감률은 약 90.91%입니다.
def m01_s3(c):
    """업무시간을 계산하고 지연이 큰 단계 찾기.

    읽을 작업값: checks, root
    저장할 작업값: workflow
    결과 폴더: folder(c, 1)
    """
    return TODO(1, 3)

# STEP 04 현재 환경을 평가하고 먼저 고칠 일 정하기
# 할 일: 각 영역에 현재 확인한 사실, 부족한 점, 담당자, 다음 작업을 적으세요. 근거가 없으면 0점, 문서나 수동 확인만 있으면 1점, 별도 시험으로 정상·오류 상황까지 확인했으면 2점을 줍니다. 점수 옆에 확인한 파일·행 또는 시험 이름을 적으세요. 앞으로 만들 계획만으로 점수를 올리지 않습니다. 코드를 한 번 실행했다는 사실만으로 오류 대응까지 검증한 것으로 보지 않습니다. 가장 먼저 고칠 작업 세 개를 고르고 순서, 담당자, 선행 작업, 완료를 확인할 조건을 적으세요. 예를 들어 코드 변환을 먼저 끝내야 품질 집계를 할 수 있다는 관계를 드러냅니다.
# 확인: 여섯 영역이 빠짐없이 있고 점수는 0~2 범위인지 확인하세요. 세 작업의 완료 조건은 “개선한다”가 아니라 어떤 파일이나 시험이 나오면 끝나는지로 적습니다.
def m01_s4(c):
    """현재 환경을 평가하고 먼저 고칠 일 정하기.

    읽을 작업값: 공통 설정
    저장할 작업값: environment_baseline
    결과 폴더: folder(c, 1)
    """
    return TODO(1, 4)

# STEP 05 다음 담당자가 사용할 변환 규칙 작성하기
# 할 일: spec.md에 입력 파일과 공통 항목의 대응, 품목·거점 코드표, 재고 이력과 출처별 관측의 식별 기준을 적으세요. 기본 단위 EA, 등록된 BOX의 10EA 환산, 시간대가 있는 시각의 UTC 변환, 60분 초과 지연 표시 방법을 예시와 함께 쓰세요. 원래 코드·값·단위·시각은 보존합니다. 결측 수량은 0으로 만들지 않고, 미등록 코드·단위·잘못된 날짜는 확인할 때까지 보류 또는 격리한다고 적으세요. 다른 두 출처가 같아도 보완 후보일 뿐 승인값이 아님을 명시합니다. 동료에게 stock_amount라는 새 항목과 출고수량이라는 새 항목을 각각 제시하세요. 전자는 재고와 같은 뜻인지 확인한 뒤 등록하고, 후자는 재고량으로 바로 연결하지 않는 이유를 검토 기록에 남깁니다.
# 확인: 파일·항목·코드·단위·시각·결측·지연·후보·승인의 기준이 모두 있는지 체크하세요. “적절히 처리한다”처럼 선택을 남기는 문장은 구체적인 조건으로 바꿉니다.
def m01_s5(c):
    """다음 담당자가 사용할 변환 규칙 작성하기.

    읽을 작업값: 공통 설정
    저장할 작업값: 근거 파일과 반환 표
    결과 폴더: folder(c, 1)
    """
    return TODO(1, 5)

result_html = run_mission(c, 1, [m01_s1, m01_s2, m01_s3, m01_s4, m01_s5])
document = result_html.read_text(encoding="utf-8")
display(HTML('<iframe title="DAT09 단계별 결과" style="width:100%;height:1100px;border:1px solid #ccd5dd" sandbox="allow-scripts allow-downloads allow-same-origin" srcdoc="'+html.escape(document,quote=True)+'"></iframe>'))
